<h1>####################################################</h1>
<h1>Enterprise Vector Store ユースケース</h1>
<h1>PDFからEnterprise Vector Storeを作成する</h1>
<h1>####################################################</h1>

# 1. 環境定義
---

## 1.1. 必要ライブラリインストール (初回オプション)

In [ ]:
!pip install -r requirements.txt

## 1.2. ライブラリ読み込み

In [ ]:
import os
from IPython.display import display, HTML 
import ipywidgets as widgets
import glob
from dotenv import load_dotenv
import panel as pn
from teradataml import *
from teradatagenai import VSManager, VectorStore
from teradataml import create_context, set_auth_token
import logging
import time
import pandas
# sqlalchemy
from sqlalchemy import text

## 1.3. データフォルダ取得

In [ ]:
cwd = os.getcwd()
data_folder = os.path.join(cwd, "data")

# 2. ベクトルデータベース操作
---

## 2.1. データベース接続

In [ ]:
env_vars = dotenv_values("env")
eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
# 接続オブジェクトを取得します
con = get_connection()
# Set Query Band
con.execute(text("""SET query_band='APP=EVS_TEST_PDF;' UPDATE FOR SESSION;"""))

## 2.2. USEに認証トークンでセッション確立

In [ ]:
if set_auth_token(base_url=env_vars.get("ues_uri"),
                  pat_token=env_vars.get("access_token"), 
                  pem_file=env_vars.get("pem_file"),
                  valid_from=int(time.time())
                 ):
    print("セッション確立の成功")
else:
    print("セッション確立の失敗")

## 2.3. Vector Store ステータスの確認

In [ ]:
VSManager.health()

## 2.4. Vector Store の指定

In [ ]:
# ベクトルデータベースがまだ作成されていない場合はワーニングが出ます
document_vector_store = VectorStore("tec_documnet")

# 3. Vector Store の作成
---
<font color="red">ベクトルストアが作成済みなら3章はスキップしてください</font>

## 3.1. アップロードされたファイルのチェック

In [ ]:
supported_patterns = ["*.pdf"]
files = []
for pattern in supported_patterns:
    files.extend(glob.glob(os.path.join(data_folder, pattern)))

if len(files) == 0:
    raise FileNotFoundError("PDFファイルが見つかりません")
else:
    print(f"### PDFファイルの一覧 ({len(files)} Files) ###")
    for file in files:
        print(os.path.basename(file))

## 3.2. ベクトルストアの作成

In [ ]:
#ベクトルストアを削除する場合は このコメントを解除してDestory()する 
document_vector_store.destroy()

In [ ]:
#ベクトルストア作成 
df=document_vector_store.status()
if df is None:
    document_vector_store.create(
        embeddings_model="amazon.titan-embed-text-v2:0",
        chat_completion_model="anthropic.claude-3-haiku-20240307-v1:0",
        search_algorithm="VECTORDISTANCE",
        top_k=10,
        object_names="ITPolicy",
        data_columns=["chunks"],
        vector_column="VectorIndex",
        optimized_chunking=True,
        document_files=files,
    )
else:
    print("ベクトルストアは既に存在しています!")

## 3.3. ステータスの確認 ***<font color="red">READY</font>*** になるまで待つ

In [ ]:
document_vector_store.status()

## 3.4. ベクトルストアのリスト表示

In [ ]:
df=VSManager.list()
df[df["permission"] == 'ADMIN']

In [ ]:
document_vector_store.get_details()

# 4. Enterprise Vector Store に質問
---

## 4. Enterprise Vector Store に質問してみる

In [ ]:
res = document_vector_store.ask(
    question="パスワードのポリシーについて教えてください",
    prompt="あなたはデータアナリストです"
)
print(res)

# 5. Vantageから切断
---

In [ ]:
remove_context()